In [ ]:
import json
from datetime import datetime

try:
    import notebookutils as nbu          # Fabric (current)
except ImportError:
    import mssparkutils as nbu           # Fabric / Synapse (legacy)

MB = 1024 * 1024

def _fs_sizes(root):
    """Recursively map file path -> size in bytes under a table folder."""
    sizes, stack = {}, [root]
    while stack:
        for f in nbu.fs.ls(stack.pop()):
            if f.isDir:
                if not f.name.startswith("_"):      # skip _delta_log
                    stack.append(f.path)
            elif f.name.endswith(".parquet"):
                sizes[f.path.split("://", 1)[-1]] = f.size
    return sizes

def table_file_stats(table_name, small_file_mb=128):
    """Return size statistics for the ACTIVE data files of a Delta table."""
    detail = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]
    location = detail["location"]

    active = {p.split("://", 1)[-1] for p in
              spark.read.format("delta").table(table_name).inputFiles()}
    all_sizes = _fs_sizes(location)
    sizes = sorted(v for k, v in all_sizes.items() if k in active)

    if not sizes:                                   # fallback if listing fails
        sizes = [detail["sizeInBytes"] // max(detail["numFiles"], 1)] * detail["numFiles"]

    total = sum(sizes)
    n = len(sizes)
    threshold = small_file_mb * MB
    return {
        "table":       table_name,
        "location":    location,
        "num_files":   n,
        "total_mb":    total / MB,
        "avg_mb":      total / n / MB,
        "min_mb":      sizes[0] / MB,
        "median_mb":   sizes[n // 2] / MB,
        "max_mb":      sizes[-1] / MB,
        "small_files": sum(1 for s in sizes if s < threshold),
        "small_pct":   100.0 * sum(1 for s in sizes if s < threshold) / n,
        "num_rows":    spark.table(table_name).count(),
        "sizes":       sizes,
    }

def print_stats(st, title):
    print("=" * 62)
    print(f"  {title}  —  {st['table']}")
    print("=" * 62)
    rows = [
        ("Data files",              f"{st['num_files']:,}"),
        ("Rows",                    f"{st['num_rows']:,}"),
        ("Total size",              f"{st['total_mb']:,.2f} MB"),
        ("Average file size",       f"{st['avg_mb']:,.2f} MB"),
        ("Median file size",        f"{st['median_mb']:,.2f} MB"),
        ("Min / Max file size",     f"{st['min_mb']:,.2f} MB / {st['max_mb']:,.2f} MB"),
        ("Small files (<128 MB)",   f"{st['small_files']:,}  ({st['small_pct']:.1f}%)"),
    ]
    for k, v in rows:
        print(f"  {k:<24} : {v:>28}")
    print("-" * 62)
    # size histogram
    buckets = [(0, 1, "< 1 MB"), (1, 8, "1–8 MB"), (8, 32, "8–32 MB"),
               (32, 128, "32–128 MB"), (128, 512, "128–512 MB"),
               (512, float("inf"), "≥ 512 MB")]
    print("  File size distribution")
    for lo, hi, label in buckets:
        c = sum(1 for s in st["sizes"] if lo * MB <= s < hi * MB)
        if c:
            print(f"    {label:<12} {c:>4}  {'█' * min(c, 40)}")
    print("=" * 62 + "\n")